# 04 商品关联规则挖掘

**目标**：找出「经常一起购买」的商品组合，指导捆绑销售和推荐策略。

核心指标：
- **Support (支持度)**：同时包含 A 和 B 的订单占比
- **Confidence (置信度)**：买了 A 的人中有多少也买了 B
- **Lift (提升度)**：A 和 B 同时出现的概率是随机情况的多少倍（>1 表示正相关）

In [2]:
import sys
sys.path.append('..')
!pip install mlxtend
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_data, clean_data
from src.association import (
    build_transaction_matrix, mine_rules,
    enrich_rules_with_category, top_rules_report, plot_rules
)

sns.set_style('whitegrid')
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

ModuleNotFoundError: No module named 'mlxtend'

In [ ]:
# 加载数据
df, item_meta = load_data(use_simulated=True, n_users=5000)
df = clean_data(df)

In [ ]:
# === 1. 构建购物篮矩阵 ===
basket_matrix = build_transaction_matrix(df, min_items=2)
print(f'购物篮矩阵形状: {basket_matrix.shape}')
print(f'有效用户数（至少购买2件）: {basket_matrix.shape[0]:,}')
print(f'涉及商品数: {basket_matrix.shape[1]:,}')

In [ ]:
# === 2. 挖掘关联规则 ===
rules = mine_rules(basket_matrix, min_support=0.02, min_confidence=0.3, min_threshold=1.0)
print(f'发现 {len(rules)} 条关联规则')

In [ ]:
# 补充品类信息
if item_meta is not None and not rules.empty:
    rules = enrich_rules_with_category(rules, item_meta)
top_rules_report(rules, top_n=15)

In [ ]:
# === 3. 可视化 ===
if not rules.empty:
    plot_rules(rules)

In [ ]:
# === 4. 输出 Top 规则到 CSV（供报告使用） ===
if not rules.empty:
    display_cols = ['antecedents', 'consequents', 'antecedents_label', 'consequents_label',
                    'support', 'confidence', 'lift', 'leverage']
    available = [c for c in display_cols if c in rules.columns]
    rules[available].head(20).to_csv('../output/top20_rules.csv', index=False)
    print('Top 20 规则已保存至 output/top20_rules.csv')

In [ ]:
# === 5. 按品类统计交叉购买 ===
if item_meta is not None and not rules.empty:
    # 统计规则中出现最多的品类对
    def get_category(items_str):
        cats = set()
        for part in items_str.split(', '):
            if '(' in part and ')' in part:
                cats.add(part.split('(')[1].rstrip(')'))
        return ', '.join(sorted(cats))
    
    rules['ante_cat'] = rules['antecedents_label'].apply(get_category)
    rules['cons_cat'] = rules['consequents_label'].apply(get_category)
    
    # 跨品类关联 Top 10
    cross_cat = rules[rules['ante_cat'] != rules['cons_cat']]
    cross_summary = cross_cat.groupby(['ante_cat', 'cons_cat']).agg(
        规则数=('lift', 'count'),
        平均Lift=('lift', 'mean')
    ).round(2).sort_values('平均Lift', ascending=False)
    print('Top 10 跨品类关联：')
    cross_summary.head(10)

---
### 本阶段结论
- 共挖掘到 X 条有效关联规则（Lift > 1）
- 最强关联：XX 和 YY（Lift = Z，即一起购买的概率是随机情况的 Z 倍）
- 跨品类洞察：XX 品类和 YY 品类存在强关联
- 业务建议：
  1. 在商品详情页推荐关联商品
  2. 将强关联商品做捆绑套餐
  3. 对买了 A 没买 B 的用户定向推送 B 的优惠券